In [7]:
import numpy as np
import xgboost as xgb
import scipy.stats as stats
import scipy.special as special
from scipy.stats import gamma, lognorm
import openturns as ot
import os
import sys

In [2]:
# Set project root relative to notebook location
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Losses import gaussian_xgb_loss as gauss
from Losses import clayton_xgb_loss as clayton
from Losses import gumbel_xgb_loss as gumbel

In [18]:
def clip_copula_samples(samples, eps=1e-10):
    """
    Ensure copula samples are strictly inside (0, 1)^2.
    
    Parameters:
        samples (np.ndarray): shape (n, 2), values in [0, 1]
        eps (float): small buffer to keep samples strictly in (0, 1)
    
    Returns:
        np.ndarray: clipped samples in (eps, 1 - eps)
    """
    return np.clip(samples, eps, 1 - eps)

def copula_to_gamma(copula_samples, mu1, mu2):
    """
    Transform copula samples to gamma margins with variable location parameters.
    
    Parameters:
        copula_samples (np.ndarray): shape (n, 2), values in (0, 1)
        mu1 (np.ndarray): shape (n,) parameters for Y1
        mu2 (np.ndarray): shape (n,) parameters for Y2
    
    Returns:
        np.ndarray: shape (n, 2), dependent gamma samples
    """
    u1, u2 = copula_samples[:, 0], copula_samples[:, 1]
    mu1 = np.asarray(mu1)
    mu2 = np.asarray(mu2)

    if np.any(u1 <= 0) or np.any(u2 <= 0):
        raise ValueError("Copula samples must be in (0, 1), not inclusive.")
    if np.any(mu1 <= 0) or np.any(mu2 <= 0):
        raise ValueError("Rate parameters must be > 0.")

    y1 = gamma.ppf(u1, a = 1.0, scale = mu1)
    y2 = gamma.ppf(u2, a = 1.0, scale = mu2) 

    return np.column_stack((y1, y2))

def copula_to_lognormal(copula_samples, mu1, mu2):
    """
    Transform copula samples to lognormal margins with variable location parameters.
    
    Parameters:
        copula_samples (np.ndarray): shape (n, 2), values in (0, 1)
        mu1 (np.ndarray): mean (n,) parameters for Y1
        mu2 (np.ndarray): mean (n,), rate parameters for Y2
    
    Returns:
        np.ndarray: shape (n, 2), dependent lognormal samples
    """
    u1, u2 = copula_samples[:, 0], copula_samples[:, 1]
    mu1 = np.asarray(mu1)
    mu2 = np.asarray(mu2)

    if np.any(u1 <= 0) or np.any(u2 <= 0):
        raise ValueError("Copula samples must be in (0, 1), not inclusive.")
    if np.any(mu1 <= 0) or np.any(mu2 <= 0):
        raise ValueError("Rate parameters must be > 0.")

    y1 = lognorm.ppf(u1, s=1.0, scale=mu1)
    y2 = lognorm.ppf(u2, s=1.0, scale=mu2) 

    return np.column_stack((y1, y2))

def copula_to_response(copula_samples, mu1, mu2):
    """
    Transform copula samples to lognormal margins with variable location parameters.
    
    Parameters:
        copula_samples (np.ndarray): shape (n, 2), values in (0, 1)
        mu1 (np.ndarray): mean (n,) parameters for Y1
        mu2 (np.ndarray): mean (n,), rate parameters for Y2
    
    Returns:
        np.ndarray: shape (n, 2), dependent lognormal samples
    """
    u1, u2 = copula_samples[:, 0], copula_samples[:, 1]
    mu1 = np.asarray(mu1)
    mu2 = np.asarray(mu2)

    if np.any(u1 <= 0) or np.any(u2 <= 0):
        raise ValueError("Copula samples must be in (0, 1), not inclusive.")
    if np.any(mu1 <= 0) or np.any(mu2 <= 0):
        raise ValueError("Rate parameters must be > 0.")

    y1 = lognorm.ppf(u1, s = 1.5, scale=mu1)
    y2 = gamma.ppf(u2, a = 1.5, scale = mu2/1.5) 

    return np.column_stack((y1, y2))

# shape of generated data.
n = 2000
p = 10

ratio = 0.7
seed = 2000

num_boost_round = 500

np.random.seed(seed=seed)

x = np.random.uniform(-1, 1, (n, p))
#x = np.random.randn(kRows, kCols)
x1 = x[:, 0]
x2 = x[:, 1]
x3 = x[:, 2]

x4 = x[:, 3]
#x5 = x[:, 4]

mu1 = np.exp(-0.75*x1 + 0.5*np.cos(np.pi*x3))
mu2 = np.exp(0.5 - 0.7*x1 - 0.02*np.exp(2*(x2 + 1)))

#mu1 = np.exp(2 + x1 - 2*x2 + 3*x3 - 4*x4 + 5*x5) #TBD
#mu2 = np.exp(2 + x1 + 2*x2 - 3*x3 + 4*x4 - 5*x5) #TBD

#theta = np.exp(0.01 - 0.2*(x1 < 0.3) + 0.2*np.sin(2*np.pi*x2) + x3**2 - x3*(np.abs(x2) > 0.5))
# First linear scenario
#tau1 = special.expit(0.5 - 0.3 * x1 + 0.4*x2 - 2*x3)
# Second non-linear scenario
#tau2 = special.expit(0.1 - 0.2*(x1 < 0.3) + 0.3*np.sin(2*np.pi*x2) + 0.5*x3**2 - 2.5*x3*(np.abs(x2) > 0.5))

tau = special.expit(-0.8 + 1.5*np.log(4.5 - 1.7*np.sin(np.pi*x4)))

In [19]:
from sklearn.metrics import make_scorer
from xgboost import XGBRegressor

gauss_scorer = make_scorer(gauss.gaussian_copula_nll_metric, greater_is_better=False)

gauss_model = XGBRegressor(
    objective=gauss.gaussian_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

clayton_scorer = make_scorer(clayton.clayton_copula_nll_metric, greater_is_better=False)

clayton_model = XGBRegressor(
    objective=clayton.clayton_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

gumbel_scorer = make_scorer(gumbel.gumbel_copula_nll_metric, greater_is_better=False)

gumbel_model = XGBRegressor(
    objective=gumbel.gumbel_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

#gamma_scorer = make_scorer(gauss.gaussian_copula_nll_metric, greater_is_better=False)

gamma_model = XGBRegressor(
    objective="reg:gamma",
    n_estimators=500,
    tree_method="hist",
    eval_metric="gamma-nloglik"
)

normal_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    tree_method="hist",
    eval_metric="rmse"
)

In [20]:
from sklearn.model_selection import train_test_split, GridSearchCV

param_grid = {
    'max_depth': [1, 2, 3, 5],
    'eta':[0.01, 0.1, 0.3],
    'lambda': [0.0, 0.1, 0.2]
    #'subsample': [0.6, 0.8, 1.0],
    #'colsample_bytree': [0.6, 0.8, 1.0]
}

grid_search_gauss=GridSearchCV(
    estimator=gauss_model,
    param_grid=param_grid,
    scoring=gauss_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_clayton=GridSearchCV(
    estimator=clayton_model,
    param_grid=param_grid,
    scoring=clayton_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_gumbel=GridSearchCV(
    estimator=gumbel_model,
    param_grid=param_grid,
    scoring=gumbel_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_gamma=GridSearchCV(
    estimator=gamma_model,
    param_grid=param_grid,
    scoring="neg_mean_gamma_deviance",
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_normal=GridSearchCV(
    estimator=normal_model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
    )

In [37]:
u_gauss = clip_copula_samples(gauss.sample_gaussian_copula(gauss.gauss_tau2par(tau), seed=seed))
y_gauss = copula_to_response(u_gauss, mu1, mu2)

x_main_gauss, _, y_main_gauss, _ = train_test_split(x, y_gauss, test_size=1-ratio, random_state=42)

grid_search_normal.fit(x_main_gauss, np.log(y_main_gauss[:, 0]))

print(f"Best parameters (lognormal_gauss): {grid_search_normal.best_params_}")
print(f"Best score (lognormal_gauss): {grid_search_normal.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (lognormal_gauss): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 1}
Best score (lognormal_gauss): -2.415286403570979


In [38]:
grid_search_gamma.fit(x_main_gauss, y_main_gauss[:, 1])

print(f"Best parameters (gamma_gauss): {grid_search_gamma.best_params_}")
print(f"Best score (gamma_gauss): {grid_search_gamma.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gamma_gauss): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 1}
Best score (gamma_gauss): -0.7893374475371626


In [39]:
normal_model.set_params(learning_rate = grid_search_normal.best_params_["eta"], max_depth = grid_search_normal.best_params_["max_depth"], reg_lambda = grid_search_normal.best_params_["lambda"])
gamma_model.set_params(learning_rate = grid_search_gamma.best_params_["eta"], max_depth = grid_search_gamma.best_params_["max_depth"], reg_lambda = grid_search_gamma.best_params_["lambda"])

x_train_gauss, x_val_gauss, y_train_gauss, y_val_gauss = train_test_split(x_main_gauss, y_main_gauss, test_size=0.125, random_state=42)

normal_xgb1 = normal_model.fit(x_train_gauss, np.log(y_train_gauss[:,0]), eval_set=[(x_val_gauss, np.log(y_val_gauss[:,0]))])
gamma_xgb2 = gamma_model.fit(x_train_gauss, y_train_gauss[:,1], eval_set=[(x_val_gauss, y_val_gauss[:,1])])
    
mu1_main_hat = normal_xgb1.predict(x_main_gauss)
mu2_main_hat = gamma_xgb2.predict(x_main_gauss)

pseudo_obs_main_gauss = clip_copula_samples(np.column_stack((lognorm.cdf(y_main_gauss[:, 0], s = 1.5, scale=np.exp(mu1_main_hat)), gamma.cdf(y_main_gauss[:, 1], a=1.5, scale=mu2_main_hat/1.5))), eps=1e-7)

[0]	validation_0-rmse:1.65350
[1]	validation_0-rmse:1.65252
[2]	validation_0-rmse:1.65157
[3]	validation_0-rmse:1.65064
[4]	validation_0-rmse:1.64992
[5]	validation_0-rmse:1.64902
[6]	validation_0-rmse:1.64833
[7]	validation_0-rmse:1.64747
[8]	validation_0-rmse:1.64681
[9]	validation_0-rmse:1.64599
[10]	validation_0-rmse:1.64536
[11]	validation_0-rmse:1.64456
[12]	validation_0-rmse:1.64396
[13]	validation_0-rmse:1.64320
[14]	validation_0-rmse:1.64263
[15]	validation_0-rmse:1.64190
[16]	validation_0-rmse:1.64135
[17]	validation_0-rmse:1.64065
[18]	validation_0-rmse:1.64013
[19]	validation_0-rmse:1.63946
[20]	validation_0-rmse:1.63896
[21]	validation_0-rmse:1.63832
[22]	validation_0-rmse:1.63784
[23]	validation_0-rmse:1.63722
[24]	validation_0-rmse:1.63677
[25]	validation_0-rmse:1.63618
[26]	validation_0-rmse:1.63575
[27]	validation_0-rmse:1.63518
[28]	validation_0-rmse:1.63477
[29]	validation_0-rmse:1.63422
[30]	validation_0-rmse:1.63384
[31]	validation_0-rmse:1.63331
[32]	validation_0-

In [40]:
grid_search_gauss.fit(x_main_gauss, pseudo_obs_main_gauss)

print(f"Best parameters (gauss): {grid_search_gauss.best_params_}")
print(f"Best score (gauss): {grid_search_gauss.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gauss): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 2}
Best score (gauss): 620.4240789186321


In [41]:
u_clayton = clip_copula_samples(clayton.sample_clayton_copula(clayton.clayton_tau2par(tau), seed=seed))
y_clayton = copula_to_response(u_clayton, mu1, mu2)

x_main_clayton, _, y_main_clayton, _ = train_test_split(x, y_clayton, test_size=1-ratio, random_state=42)

grid_search_normal.fit(x_main_clayton, np.log(y_main_clayton[:, 0]))

print(f"Best parameters (lognormal_clayton): {grid_search_normal.best_params_}")
print(f"Best score (lognormal_clayton): {grid_search_normal.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (lognormal_clayton): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 1}
Best score (lognormal_clayton): -2.229492967864876


In [42]:
grid_search_gamma.fit(x_main_clayton, y_main_clayton[:, 1])

print(f"Best parameters (gamma_clayton): {grid_search_gamma.best_params_}")
print(f"Best score (gamma_clayton): {grid_search_gamma.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gamma_clayton): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 1}
Best score (gamma_clayton): -0.7308146031051701


In [43]:
normal_model.set_params(learning_rate = grid_search_normal.best_params_["eta"], max_depth = grid_search_normal.best_params_["max_depth"], reg_lambda = grid_search_normal.best_params_["lambda"])
gamma_model.set_params(learning_rate = grid_search_gamma.best_params_["eta"], max_depth = grid_search_gamma.best_params_["max_depth"], reg_lambda = grid_search_gamma.best_params_["lambda"])

x_train_clayton, x_val_clayton, y_train_clayton, y_val_clayton = train_test_split(x_main_clayton, y_main_clayton, test_size=0.125, random_state=42)

normal_xgb1 = normal_model.fit(x_train_clayton, np.log(y_train_clayton[:,0]), eval_set=[(x_val_clayton, np.log(y_val_clayton[:,0]))])
gamma_xgb2 = gamma_model.fit(x_train_clayton, y_train_clayton[:,1], eval_set=[(x_val_clayton, y_val_clayton[:,1])])
    
mu1_main_hat = normal_xgb1.predict(x_main_clayton)
mu2_main_hat = gamma_xgb2.predict(x_main_clayton)

pseudo_obs_main_clayton = clip_copula_samples(np.column_stack((lognorm.cdf(y_main_clayton[:, 0], s = 1.5, scale=np.exp(mu1_main_hat)), gamma.cdf(y_main_clayton[:, 1], a=1.5, scale=mu2_main_hat/1.5))), eps=1e-7)

[0]	validation_0-rmse:1.54462
[1]	validation_0-rmse:1.54379
[2]	validation_0-rmse:1.54296
[3]	validation_0-rmse:1.54216
[4]	validation_0-rmse:1.54137
[5]	validation_0-rmse:1.54060
[6]	validation_0-rmse:1.53984
[7]	validation_0-rmse:1.53910
[8]	validation_0-rmse:1.53838
[9]	validation_0-rmse:1.53767
[10]	validation_0-rmse:1.53698
[11]	validation_0-rmse:1.53630
[12]	validation_0-rmse:1.53563


[13]	validation_0-rmse:1.53498
[14]	validation_0-rmse:1.53435
[15]	validation_0-rmse:1.53372
[16]	validation_0-rmse:1.53311
[17]	validation_0-rmse:1.53252
[18]	validation_0-rmse:1.53193
[19]	validation_0-rmse:1.53136
[20]	validation_0-rmse:1.53080
[21]	validation_0-rmse:1.53025
[22]	validation_0-rmse:1.52972
[23]	validation_0-rmse:1.52919
[24]	validation_0-rmse:1.52868
[25]	validation_0-rmse:1.52818
[26]	validation_0-rmse:1.52768
[27]	validation_0-rmse:1.52720
[28]	validation_0-rmse:1.52673
[29]	validation_0-rmse:1.52627
[30]	validation_0-rmse:1.52582
[31]	validation_0-rmse:1.52533
[32]	validation_0-rmse:1.52490
[33]	validation_0-rmse:1.52442
[34]	validation_0-rmse:1.52401
[35]	validation_0-rmse:1.52355
[36]	validation_0-rmse:1.52310
[37]	validation_0-rmse:1.52272
[38]	validation_0-rmse:1.52234
[39]	validation_0-rmse:1.52192
[40]	validation_0-rmse:1.52150
[41]	validation_0-rmse:1.52115
[42]	validation_0-rmse:1.52080
[43]	validation_0-rmse:1.52042
[44]	validation_0-rmse:1.52004
[45]	val

In [44]:
grid_search_clayton.fit(x_main_clayton, pseudo_obs_main_clayton)

print(f"Best parameters (clayton): {grid_search_clayton.best_params_}")
print(f"Best score (clayton): {grid_search_clayton.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (clayton): {'eta': 0.01, 'lambda': 0.2, 'max_depth': 1}
Best score (clayton): 586.0096453040676


In [45]:
u_gumbel = clip_copula_samples(gumbel.sample_gumbel_copula(gumbel.gumbel_tau2par(tau), seed=seed))
y_gumbel = copula_to_response(u_gumbel, mu1, mu2)

x_main_gumbel, _, y_main_gumbel, _ = train_test_split(x, y_gumbel, test_size=1-ratio, random_state=42)

grid_search_normal.fit(x_main_gumbel, np.log(y_main_gumbel[:, 0]))

print(f"Best parameters (lognormal_gumbel): {grid_search_normal.best_params_}")
print(f"Best score (lognormal_gumbel): {grid_search_normal.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (lognormal_gumbel): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 2}
Best score (lognormal_gumbel): -2.2579646100668738


In [46]:
grid_search_gamma.fit(x_main_gumbel, y_main_gumbel[:, 1])

print(f"Best parameters (gamma_gumbel): {grid_search_gamma.best_params_}")
print(f"Best score (gamma_gumbel): {grid_search_gamma.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


Best parameters (gamma_gumbel): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 1}
Best score (gamma_gumbel): -0.7511196062591592


In [47]:
normal_model.set_params(learning_rate = grid_search_normal.best_params_["eta"], max_depth = grid_search_normal.best_params_["max_depth"], reg_lambda = grid_search_normal.best_params_["lambda"])
gamma_model.set_params(learning_rate = grid_search_gamma.best_params_["eta"], max_depth = grid_search_gamma.best_params_["max_depth"], reg_lambda = grid_search_gamma.best_params_["lambda"])

x_train_gumbel, x_val_gumbel, y_train_gumbel, y_val_gumbel = train_test_split(x_main_gumbel, y_main_gumbel, test_size=0.125, random_state=42)

normal_xgb1 = normal_model.fit(x_train_gumbel, np.log(y_train_gumbel[:,0]), eval_set=[(x_val_gumbel, np.log(y_val_gumbel[:,0]))])
gamma_xgb2 = gamma_model.fit(x_train_gumbel, y_train_gumbel[:,1], eval_set=[(x_val_gumbel, y_val_gumbel[:,1])])
    
mu1_main_hat = normal_xgb1.predict(x_main_gumbel)
mu2_main_hat = gamma_xgb2.predict(x_main_gumbel)

pseudo_obs_main_gumbel = clip_copula_samples(np.column_stack((lognorm.cdf(y_main_gumbel[:, 0], s = 1.5, scale=np.exp(mu1_main_hat)), gamma.cdf(y_main_gumbel[:, 1], a=1.5, scale=mu2_main_hat/1.5))), eps=1e-7)

[0]	validation_0-rmse:1.50220
[1]	validation_0-rmse:1.50163
[2]	validation_0-rmse:1.50108
[3]	validation_0-rmse:1.50054
[4]	validation_0-rmse:1.50001
[5]	validation_0-rmse:1.49949
[6]	validation_0-rmse:1.49898
[7]	validation_0-rmse:1.49865
[8]	validation_0-rmse:1.49824
[9]	validation_0-rmse:1.49783
[10]	validation_0-rmse:1.49744
[11]	validation_0-rmse:1.49715
[12]	validation_0-rmse:1.49677
[13]	validation_0-rmse:1.49633
[14]	validation_0-rmse:1.49597
[15]	validation_0-rmse:1.49549
[16]	validation_0-rmse:1.49514
[17]	validation_0-rmse:1.49477
[18]	validation_0-rmse:1.49432
[19]	validation_0-rmse:1.49392
[20]	validation_0-rmse:1.49361
[21]	validation_0-rmse:1.49340
[22]	validation_0-rmse:1.49309
[23]	validation_0-rmse:1.49267
[24]	validation_0-rmse:1.49235
[25]	validation_0-rmse:1.49199
[26]	validation_0-rmse:1.49160
[27]	validation_0-rmse:1.49130
[28]	validation_0-rmse:1.49096
[29]	validation_0-rmse:1.49059
[30]	validation_0-rmse:1.49045
[31]	validation_0-rmse:1.49017
[32]	validation_0-

In [48]:
grid_search_gumbel.fit(x_main_gumbel, pseudo_obs_main_gumbel)

print(f"Best parameters (gumbel): {grid_search_gumbel.best_params_}")
print(f"Best score (gumbel): {grid_search_gumbel.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


Best parameters (gumbel): {'eta': 0.01, 'lambda': 0.2, 'max_depth': 2}
Best score (gumbel): 528.4927089155244
